# SDG Classification of Web of Science (WoS) Records

This notebook classifies research papers (exported from Web of Science) against the **17 UN Sustainable Development Goals (SDGs)** using keyword matching on the paper's Title, Abstract, and Keywords. It then:

1. Loads and validates the input data
2. Classifies each paper against one or more SDGs
3. Produces summary statistics and a bar-chart of SDG coverage
4. Builds a bipartite paper↔SDG network graph and exports it (GraphML) for tools like Gephi
5. Renders a quick in-notebook network visualization

> **Tip:** Update the `DATA_DIR` path in the next cell to point at the folder containing your WoS export (`output_wos_file.csv`).

In [ ]:
# --- Make the shared `techintel` package importable ---------------------------
# The WoS parsing / graph / metrics helpers used to be copy-pasted into each
# notebook. They now live in techintel/ at the repository root; this finds that
# root by walking up from the working directory.
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "techintel" / "__init__.py").exists():
            return candidate
    raise RuntimeError(
        "Could not find the repository root (a directory containing 'techintel/')."
    )


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

# ============================================================
# CONFIGURATION - edit these paths/columns as needed
# ============================================================
# Relative to the notebook's working directory. This was previously hard-coded
# to an absolute r"G:\My Drive\..." path, which only worked on one machine.
DATA_DIR = Path.cwd() / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILE      = DATA_DIR / "output_wos_file.csv"       # raw WoS export
CLASSIFIED_FILE = DATA_DIR / "wos_sdg_results.csv"        # classified output
GRAPH_FILE      = DATA_DIR / "wos_sdg_graph.graphml"      # bipartite graph export

# Columns to search for SDG keywords. Any that are missing in the
# source file are simply skipped (with a warning) rather than failing.
TEXT_COLUMNS = ["Title", "Abstract", "Keywords", "Author Keywords"]

print(f"Data directory : {DATA_DIR}")
print(f"Input file     : {INPUT_FILE}")


## SDG Keyword Dictionary

An expanded keyword list per SDG (broader coverage than a minimal set). Feel free to tune these lists for your domain (e.g. water/materials science).

In [ ]:
sdg_keywords = {
    "SDG 1: No Poverty": [
        "poverty", "extreme poverty", "income inequality", "social protection",
        "poverty reduction", "poverty alleviation", "low-income", "vulnerable population",
        "basic needs", "social safety net"
    ],
    "SDG 2: Zero Hunger": [
        "food security", "food insecurity", "nutrition", "malnutrition", "agriculture",
        "hunger", "sustainable agriculture", "crop yield", "food production",
        "smallholder farmer", "agricultural productivity", "soil fertility"
    ],
    "SDG 3: Good Health and Well-being": [
        "healthcare", "health care", "disease", "medicine", "well-being", "wellbeing",
        "public health", "mental health", "infectious disease", "vaccination",
        "maternal health", "mortality rate", "epidemiology", "clinical trial"
    ],
    "SDG 4: Quality Education": [
        "education", "literacy", "schooling", "training", "higher education",
        "vocational training", "lifelong learning", "e-learning", "curriculum",
        "student achievement", "educational attainment"
    ],
    "SDG 5: Gender Equality": [
        "gender equality", "women empowerment", "female participation", "gender gap",
        "gender-based violence", "women's rights", "gender parity", "female leadership",
        "gender discrimination"
    ],
    "SDG 6: Clean Water and Sanitation": [
        "clean water", "sanitation", "wastewater", "hygiene", "drinking water",
        "water quality", "water treatment", "water scarcity", "water resources",
        "water pollution", "water management", "groundwater", "desalination"
    ],
    "SDG 7: Affordable and Clean Energy": [
        "renewable energy", "solar", "wind energy", "clean energy", "energy efficiency",
        "photovoltaic", "biomass energy", "hydropower", "energy storage",
        "energy access", "sustainable energy", "fossil fuel"
    ],
    "SDG 8: Decent Work and Economic Growth": [
        "employment", "economic growth", "labor market", "decent work",
        "job creation", "unemployment", "economic development", "labor rights",
        "productivity growth", "gross domestic product", "informal economy"
    ],
    "SDG 9: Industry, Innovation and Infrastructure": [
        "innovation", "infrastructure", "materials science", "manufacturing",
        "industrialization", "research and development", "technology transfer",
        "supply chain", "advanced materials", "industry 4.0", "nanotechnology"
    ],
    "SDG 10: Reduced Inequalities": [
        "inequality", "social justice", "migration", "income disparity",
        "social inclusion", "discrimination", "marginalized communities",
        "equal opportunity", "wealth distribution"
    ],
    "SDG 11: Sustainable Cities and Communities": [
        "urbanization", "smart cities", "housing", "transport", "urban planning",
        "public transportation", "urban development", "resilient cities",
        "slum", "land use planning", "air quality"
    ],
    "SDG 12: Responsible Consumption and Production": [
        "sustainable production", "recycling", "circular economy", "waste reduction",
        "waste management", "sustainable consumption", "resource efficiency",
        "life cycle assessment", "eco-design", "upcycling", "single-use plastic"
    ],
    "SDG 13: Climate Action": [
        "climate change", "carbon emissions", "adaptation", "mitigation",
        "greenhouse gas", "carbon footprint", "global warming", "climate resilience",
        "carbon capture", "net zero", "decarbonization"
    ],
    "SDG 14: Life Below Water": [
        "marine", "ocean", "fisheries", "biodiversity", "coral reef",
        "ocean acidification", "marine ecosystem", "overfishing",
        "marine pollution", "coastal ecosystem", "aquaculture"
    ],
    "SDG 15: Life on Land": [
        "forests", "biodiversity", "ecosystems", "land degradation", "deforestation",
        "reforestation", "soil erosion", "wildlife conservation", "habitat loss",
        "terrestrial ecosystem", "desertification"
    ],
    "SDG 16: Peace, Justice and Strong Institutions": [
        "governance", "justice", "human rights", "institutions", "corruption",
        "rule of law", "transparency", "accountability", "peacebuilding",
        "conflict resolution", "civil society"
    ],
    "SDG 17: Partnerships for the Goals": [
        "international cooperation", "partnerships", "global collaboration",
        "public-private partnership", "capacity building", "development finance",
        "technology sharing", "multilateral cooperation"
    ]
}

# Pre-compile regex patterns once for performance (word-boundary, case-insensitive)
sdg_patterns = {
    sdg: [re.compile(r"\b" + re.escape(kw) + r"\b", re.IGNORECASE) for kw in kws]
    for sdg, kws in sdg_keywords.items()
}

print(f"Loaded keyword dictionary for {len(sdg_keywords)} SDGs "
      f"({sum(len(v) for v in sdg_keywords.values())} keywords total).")

## Load & Validate Data

Loads the WoS CSV with basic error handling: missing file, encoding issues, empty data, and missing expected columns are all reported clearly instead of raising an obscure exception.

In [ ]:
from techintel import wos

# `load_wos_file` is now `techintel.wos.read_wos_csv`: it tries utf-8, utf-8-sig
# and latin1, warns about expected-but-missing columns, and raises a clear error
# when the file is absent, empty, or has none of the expected columns.
df = wos.read_wos_csv(INPUT_FILE, expected_columns=TEXT_COLUMNS)
df.head()


## Classify Papers Against SDGs

Each paper's combined text (Title + Abstract + Keywords, whichever are available) is scanned for SDG-related keywords. The classifier returns:

- `SDG_matches`: list of all matched SDGs
- `SDG_match_count`: number of keyword hits per SDG (used for a simple "primary SDG" ranking)
- `Primary_SDG`: the SDG with the most keyword hits (or "No clear SDG match")

In [ ]:
def build_combined_text(row, columns) -> str:
    parts = [str(row[c]) for c in columns if c in row and pd.notna(row[c])]
    return " ".join(parts).lower()


def classify_sdg(text: str):
    """Return (matched_sdgs, hit_counts_dict) for a given text."""
    if not text or pd.isna(text):
        return [], {}

    hit_counts = {}
    for sdg, patterns in sdg_patterns.items():
        count = sum(1 for pat in patterns if pat.search(text))
        if count > 0:
            hit_counts[sdg] = count

    matched = list(hit_counts.keys())
    return matched, hit_counts


available_columns = [c for c in TEXT_COLUMNS if c in df.columns]

combined_text = df.apply(lambda row: build_combined_text(row, available_columns), axis=1)
classification_results = combined_text.apply(classify_sdg)

df["SDG_matches"] = classification_results.apply(lambda r: r[0] if r[0] else ["No clear SDG match"])
df["SDG_hit_counts"] = classification_results.apply(lambda r: r[1])
df["Primary_SDG"] = df["SDG_hit_counts"].apply(
    lambda d: max(d, key=d.get) if d else "No clear SDG match"
)

n_matched = (df["Primary_SDG"] != "No clear SDG match").sum()
print(f"Classified {len(df):,} papers — {n_matched:,} matched at least one SDG "
      f"({n_matched / len(df):.1%}).")

df.to_csv(CLASSIFIED_FILE, index=False)
print(f"Saved classified results to: {CLASSIFIED_FILE}")
df.head()

## Summary Statistics

Counts and percentage of papers matching each SDG (a paper can match more than one SDG, so percentages need not sum to 100%).

In [ ]:
sdg_counter = Counter()
for matches in df["SDG_matches"]:
    for sdg in matches:
        if sdg != "No clear SDG match":
            sdg_counter[sdg] += 1

total_papers = len(df)
summary_df = (
    pd.DataFrame(sdg_counter.items(), columns=["SDG", "Paper Count"])
    .assign(**{"Percent of Papers": lambda d: (d["Paper Count"] / total_papers * 100).round(1)})
    .sort_values("Paper Count", ascending=False)
    .reset_index(drop=True)
)

no_match_count = (df["Primary_SDG"] == "No clear SDG match").sum()
print(f"Total papers            : {total_papers:,}")
print(f"Papers with no SDG match: {no_match_count:,} ({no_match_count/total_papers:.1%})")
summary_df

## SDG Distribution Bar Chart

In [ ]:
if not summary_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(summary_df["SDG"], summary_df["Paper Count"], color="#2E86AB")
    ax.set_xlabel("Number of Papers")
    ax.set_title("Paper Count by SDG")
    ax.invert_yaxis()  # highest count at top
    for i, v in enumerate(summary_df["Paper Count"]):
        ax.text(v + 0.3, i, str(v), va="center")
    plt.tight_layout()
    chart_path = DATA_DIR / "sdg_distribution.png"
    plt.savefig(chart_path, dpi=150)
    plt.show()
    print(f"Chart saved to: {chart_path}")
else:
    print("No SDG matches found — nothing to plot.")

## Build Paper ↔ SDG Bipartite Graph

Builds a graph connecting each paper to its matched SDG(s), excluding papers with no clear match, and exports it as GraphML for use in tools like Gephi.

In [ ]:
graph_df = df[df["SDG_matches"].apply(lambda matches: "No clear SDG match" not in matches)]

G = nx.Graph()

for idx, row in graph_df.iterrows():
    paper_id = f"Paper_{idx}"
    title = row.get("Title", f"Paper {idx}")
    G.add_node(paper_id, type="paper", title=str(title))

    for sdg in row["SDG_matches"]:
        G.add_node(sdg, type="sdg")
        G.add_edge(paper_id, sdg)

print(f"Graph built: {G.number_of_nodes():,} nodes "
      f"({sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'paper'):,} papers, "
      f"{sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'sdg'):,} SDGs), "
      f"{G.number_of_edges():,} edges.")

nx.write_graphml(G, GRAPH_FILE)
print(f"GraphML file created: {GRAPH_FILE}")

## Quick Network Visualization

A lightweight in-notebook preview of the paper↔SDG network (SDG nodes highlighted). For large datasets, prefer exploring the GraphML file in Gephi instead.

In [ ]:
if G.number_of_nodes() > 0:
    MAX_NODES_TO_PLOT = 500  # guard against overly dense/slow plots
    if G.number_of_nodes() > MAX_NODES_TO_PLOT:
        print(f"Graph has {G.number_of_nodes():,} nodes — plotting only the SDG nodes "
              f"plus a sample of connected papers for readability.")
        sdg_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "sdg"]
        paper_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "paper"]
        sample_papers = paper_nodes[: max(0, MAX_NODES_TO_PLOT - len(sdg_nodes))]
        plot_graph = G.subgraph(sdg_nodes + sample_papers)
    else:
        plot_graph = G

    node_colors = [
        "#E63946" if d.get("type") == "sdg" else "#A8DADC"
        for _, d in plot_graph.nodes(data=True)
    ]
    node_sizes = [
        600 if d.get("type") == "sdg" else 30
        for _, d in plot_graph.nodes(data=True)
    ]

    plt.figure(figsize=(12, 10))
    pos = nx.spring_layout(plot_graph, k=0.3, seed=42)
    nx.draw_networkx_edges(plot_graph, pos, alpha=0.2)
    nx.draw_networkx_nodes(plot_graph, pos, node_color=node_colors, node_size=node_sizes)

    sdg_labels = {n: n.split(":")[0] for n, d in plot_graph.nodes(data=True) if d.get("type") == "sdg"}
    nx.draw_networkx_labels(plot_graph, pos, labels=sdg_labels, font_size=9, font_weight="bold")

    plt.title("Paper ↔ SDG Network (red = SDG, blue = paper)")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Graph is empty — nothing to visualize.")